In [ ]:
%load_ext autoreload
%autoreload 2

# Global Scale Figure

Two-panel figure (single HTML output):
1. **Left — Orthographic globe**: `projects/globe_panel.py` — Plotly `Scattergeo` with two GCM grid resolutions
2. **Right — Time×Space diagram**: `prime_monthly.py` pipeline with `set_color_by_category`

No matplotlib or cartopy required.

**Run from the timeSpace repo root.**


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import numpy as np
import pandas as pd
from time import time

import plotly.io as pio

from bokeh.plotting import figure
from bokeh.embed import components
from bokeh.resources import CDN
from bokeh.models import Span, Label

from timeSpace.plotting import (
    create_space_time_figure,
    add_magnitude_labels,
    add_processes,
    add_legend,
)
from timeSpace.etl import transform_process_response_sheet

from globe_panel import create_globe_figure, create_scale_cascade_html


## 1. Category color mapping

In [2]:
CATEGORY_COLORS = {
    # Bright, clearly distinguishable colors by process type
    "Physical":   "#3A7EC8",   # blue
    "Biological": "#4CAF6E",   # green
    "Chemical":   "#D4732A",   # amber
}

CATEGORY_ORDER = {"Physical": 0, "Biological": 1, "Chemical": 2}


## 2. Load and transform (label offsets from CSV)


In [3]:
def transform_data(config_data):
    """Load process CSV, parse magnitudes, and prepare for plotting.

    Label positioning (label_side, x_offset, y_offset, label_text) is read
    directly from the CSV.  When a column is missing or a value is blank,
    defaults are: label_side='left', offsets=0, label_text=Process name.
    """
    data = pd.read_csv("../data/local_data/stommel_boyd2015_volumes.csv", keep_default_na=False)
    data.dropna(subset=["Time_min", "Time_max", "Space_min", "Space_max"], inplace=True)
    data.rename(columns={"Process": "ShortName"}, inplace=True)
    data["Prefix"] = data["ShortName"].str[:3]

    columns = ["Prefix", "ShortName", "Time_min", "Time_max", "Space_min", "Space_max",
               "Color", "ProcessType"]
    transformed_data = transform_process_response_sheet(data, possible_col_list=columns)

    # ── Label positioning from CSV ────────────────────────────────────────
    transformed_data["label_side"] = data["label_side"].where(
        data.get("label_side", pd.Series(dtype=str)).isin(["left", "right"]), "left"
    )
    transformed_data["x_offset"] = pd.to_numeric(data.get("x_offset", 0), errors="coerce").fillna(0).astype(int)
    transformed_data["y_offset"] = pd.to_numeric(data.get("y_offset", 0), errors="coerce").fillna(0).astype(int)

    raw_label = data.get("label_text", pd.Series(dtype=str))
    transformed_data["label_text"] = [
        lt if lt else sn
        for lt, sn in zip(raw_label, transformed_data["ShortName"])
    ]

    # Sort by ProcessType order then name so legend groups are contiguous
    transformed_data["_cat_order"] = transformed_data["ProcessType"].map(CATEGORY_ORDER).fillna(99)
    transformed_data = transformed_data.sort_values(["_cat_order", "ShortName"]).drop(columns=["_cat_order"])
    transformed_data = transformed_data.reset_index(drop=True)
    return transformed_data


config = {
    "title": "",
    "output_file": "modeled-processes-category-colors",
    "concept_name": "Process",
}

transformed_data = transform_data(config)
transformed_data[["ShortName", "ProcessType", "label_side", "x_offset", "y_offset"]]


,ShortName,ProcessType,label_side,x_offset,y_offset
0,Biological pump,Models,right,-95,-95
1,Blooms,Models,left,0,0
2,cellular behavior & microbial interactions,Experiments,left,0,0
3,cellular traits & metabolic networks,Experiments,left,0,0
4,marine snow particles,Experiments,left,0,0
5,"microbial rates, growth & death",Experiments,left,0,0
6,planetary-scale model,Models,left,0,0


## 3. Time×Space diagram

In [4]:
LABEL_FONT_SIZE = "10pt"   # screen rendering; plotting.py default (20pt) is for print/poster
MARKER_FONT_SIZE = "9pt"   # time/space marker labels

def make_stommel_panel(transformed_data, config_data):
    """
    Build the Bokeh Stommel diagram.

    interactive=False → all processes start visible.
    click_policy='hide' lets users hide individual processes via the legend.
    font_size passed explicitly so screen output is readable without being huge.
    """
    p = create_space_time_figure(width=1400, height=900, title=config_data["title"])
    p = add_magnitude_labels(p, font_size=MARKER_FONT_SIZE)
    import math as _math
    import numpy as _np

    p = add_processes(p, transformed_data, interactive=False, font_size=LABEL_FONT_SIZE, label_side="left",
                      category_col="ProcessType", category_colors=CATEGORY_COLORS)
    p = add_legend(p)

    # Override poster-scale axis fonts for screen
    p.axis.axis_label_text_font_size = "14pt"
    p.axis.major_label_text_font_size = "12pt"
    p.title.text_font_size = "14pt"
    p.legend.label_text_font_size = "11pt"
    p.legend.glyph_height = 14
    p.legend.glyph_width  = 14
    p.legend.spacing = 3

    return p


## 4. Compose and save

We build the output HTML ourselves using:
- `pio.to_html(full_html=False)` for the Plotly globe
- `components()` for the Bokeh Stommel plot (script + div fragment)
- A hand-rolled HTML wrapper that loads both CDNs

This bypasses Bokeh's browser-side HTML sanitizer, which strips `<script>` tags
from `Div` models.

In [5]:
# Guard: re-run transform if cells were executed out of order
if 'config' not in dir():
    config = {
        "title": "Modeled Processes",
        "output_file": "modeled-processes-category-colors",
        "concept_name": "Process",
    }
if 'transformed_data' not in dir():
    transformed_data = transform_data(config)

STOMMEL_WIDTH  = 1400
STOMMEL_HEIGHT =  900
GLOBE_WIDTH    =  700
GLOBE_HEIGHT   =  700

p_stommel    = make_stommel_panel(transformed_data, config)
fig_globe    = create_globe_figure(
    category_colors=CATEGORY_COLORS,
    width=GLOBE_WIDTH,
    height=GLOBE_HEIGHT,
)
cascade_html = create_scale_cascade_html(
    category_colors=CATEGORY_COLORS,
    globe_height=GLOBE_HEIGHT,
)

bokeh_script, bokeh_div = components(p_stommel)
globe_div = pio.to_html(fig_globe, full_html=False, include_plotlyjs="cdn",
                        config={"displayModeBar": False})
bokeh_js = "\n".join(f'<script src="{u}"></script>' for u in CDN.js_files)

FINE = CATEGORY_COLORS["Physical"]
arrow_svg = (
    f'<svg style="position:absolute;top:0;left:0;pointer-events:none"'
    f' width="{GLOBE_WIDTH + 60}" height="{GLOBE_HEIGHT}" xmlns="http://www.w3.org/2000/svg">'
    '<defs><marker id="ah-b" markerWidth="9" markerHeight="7" refX="9" refY="3.5" orient="auto">'
    f'<polygon points="0 0,9 3.5,0 7" fill="{FINE}"/></marker></defs>'
    f'<path d="M 272,252 C 390,252 390,120 {GLOBE_WIDTH},120"'
    f' stroke="{FINE}" stroke-width="2.2" fill="none" marker-end="url(#ah-b)"/>'
    '<text x="370" y="178" fill="#888" font-size="10" font-style="italic"'
    ' font-family="Arial" text-anchor="middle">&#xD7; 1/1000</text></svg>'
)

html = "".join([
    "<!DOCTYPE html><html lang=\"en\"><head><meta charset=\"utf-8\">",
    f"<title>Modeled Processes</title>{bokeh_js}</head><body>",
    f'<div style="display:flex;align-items:flex-start;gap:24px;padding:20px;flex-wrap:wrap;">',
    f'<div style="flex:0 0 {STOMMEL_WIDTH}px;">{bokeh_div}</div>',
    '<div style="position:relative;display:flex;flex-direction:row;align-items:flex-start;gap:0;">',
    globe_div, cascade_html, arrow_svg,
    "</div></div>",
    bokeh_script, "</body></html>",
])

file_name = f"{int(time())}-{config['output_file']}"
out_path = f"../saved_plots/{file_name}.html"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(html)
print(f"Saved: {out_path}")


Saved: ../saved_plots/1772838904-modeled-processes-category-colors.html
